In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

input_file = './output.dat'

# read binary data
with open(input_file, 'rb') as f:
    nframes = int.from_bytes(f.read(4), byteorder='little', signed=True)
    nx = int.from_bytes(f.read(4), byteorder='little', signed=True)
    ny = int.from_bytes(f.read(4), byteorder='little', signed=True)
    lon = np.frombuffer(f.read(4*nx), dtype=np.float32)
    lat = np.frombuffer(f.read(4*ny), dtype=np.float32)
    time = np.frombuffer(f.read(4*nframes), dtype=np.float32)
    performance = np.frombuffer(f.read(4*nframes), dtype=np.float32)
    elevation = np.frombuffer(f.read(4*nx*ny), dtype=np.float32).reshape((ny, nx))
    etas = np.frombuffer(f.read(4*nx*ny*nframes), dtype=np.float32).reshape((nframes, ny, nx))

# Find Krakatau location
krakatau_lon = 105.423
krakatau_lat = -6.102

j_krakatau = np.argmin(np.abs(lon - krakatau_lon))
i_krakatau = np.argmin(np.abs(lat - krakatau_lat))

In [ ]:
# Make a figure of the surface elevation map (symlog scale)
fig, ax = plt.subplots(figsize=(8, 6))
# Get colour map used by GEBCO for bathymetry (terrain) and use symmetric log scaling
extent = [lon.min(), lon.max(), lat.min(), lat.max()]
norm = mpl.colors.SymLogNorm(linthresh=10, linscale=1.0, vmin=-4435, vmax=4435)
im = ax.imshow(elevation, cmap='terrain', norm=norm, extent=extent, origin='lower')
ax.contour(lon, lat, elevation, levels=[0.5], colors='black', linewidths=0.5)
ax.scatter([krakatau_lon], [krakatau_lat], c='red', s=100, marker='*', edgecolors='black', zorder=10)
ax.set_title('Surface elevation (m)')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
fig.colorbar(im, ax=ax, label='Elevation (m)')

In [ ]:
# Plot performance of code in nano-seconds per grid point update
# Skip first point since it may be an outlier due to compilation of numba / CuPy functions on first call
plt.figure(figsize=(8, 4))
plt.plot(time[1:], performance[1:], marker='o')
plt.plot(time[1:], [np.mean(performance[1:])] * len(time[1:]), 'r--', label='Mean performance')
plt.xlabel('Time (s)')
plt.ylabel('Performance (ns per grid point update)')
plt.title('Performance of Krakatau Simulation')
plt.grid()
plt.show()
print(f'Mean performance: {np.mean(performance[1:]):.2f} ns per grid point update')

In [ ]:
# Animation: overlay eta on bathymetry for all frames
vmax = np.percentile(np.abs(etas), 99.9)
extent = [lon[0], lon[-1], lat[0], lat[-1]]

fig, ax = plt.subplots(figsize=(14, 8))

# Colormap for elevation with transparent colors
alpha_terrain = 0.3
terrain_cm_alpha = mpl.colors.ListedColormap(
        np.column_stack([mpl.colormaps['terrain'](np.linspace(0, 1, 256))[:, :3], np.full(256, alpha_terrain)])
    )

# Background: bathymetry/topography with symlog scaling
norm_elev = mpl.colors.SymLogNorm(linthresh=10, linscale=1.0, vmin=-4500, vmax=4500)
im_elev = ax.imshow(elevation, cmap=terrain_cm_alpha, norm=norm_elev, extent=extent, origin='lower')
cbar_elev = fig.colorbar(im_elev, label='Elevation (m)', ax=ax, fraction=0.046, pad=0.04)

# Add coastline contour and star for Krakatau
contours = ax.contour(lon, lat, elevation, levels=[0.5], colors='black', linewidths=0.5)
ax.scatter([krakatau_lon], [krakatau_lat], c='yellow', s=100, marker='*', edgecolors='black', zorder=10)
ax.set_xlabel('Longitude (°E)')
ax.set_ylabel('Latitude (°N)')

# Eta and title will be updated per frame
alpha_water = np.array([min(np.abs((i-128)/10),1) for i in range(256)]) # alpha goes from 0 at center to 1 at 20 steps from center
water_cm_alpha = mpl.colors.ListedColormap(
        np.column_stack([mpl.colormaps['RdBu_r'](np.linspace(0, 1, 256))[:, :3], alpha_water])
    )

im_eta = ax.imshow(etas[0], extent=extent, origin='lower', cmap=water_cm_alpha, vmin=-vmax, vmax=vmax)
cbar_eta = fig.colorbar(im_eta, ax=ax, label='Water surface elevation η (m)', fraction=0.046, pad=0.10)
ax.set_title(f'Water Surface Elevation at t = {time[0]:.0f}s ({time[0]/60:.1f} min)')

def animate(frame):
    im_eta.set_data(etas[frame])
    ax.set_title(f'Water Surface Elevation at t = {time[frame]:.0f}s ({time[frame]/60:.1f} min)')
    return [im_eta]

# Create animation
plt.rcParams['animation.embed_limit'] = 200  # memory limit in MB for embedded animation
anim = FuncAnimation(fig, animate, frames=nframes, interval=200, blit=False)
plt.close()

# Display animation
HTML(anim.to_jshtml())